# Exploratory Data Analysis

In [ ]:
from pathlib import Path

import duckdb

# Connect to in-memory DuckDB engine
con = duckdb.connect()

BASE_DIR = Path.cwd().resolve().parent.parent
DATA_DIR = BASE_DIR / "data" / "sample"

print(BASE_DIR)
print(DATA_DIR)

## Inspection of the three tables

### Call center interactions table

In [ ]:
# Inspect call_center_interactions schema
con.sql(
    f"DESCRIBE SELECT * FROM '{DATA_DIR}/call_center_interactions/*.csv' LIMIT 1"
).show()

In [ ]:
interactions_df = con.sql(
    f"SELECT * FROM '{DATA_DIR}/call_center_interactions/*.csv' "
).to_df()

In [ ]:
interactions_df.info()

In [ ]:
interactions_df.interaction_type.value_counts()

In [ ]:
interactions_df.reason_category.unique()

In [ ]:
interactions_df.contact_reason.unique()

In [ ]:
# Every pair of columns in contact_reason and reason_category
# contain the same value for each row
all(interactions_df.contact_reason == interactions_df.reason_category)

In [ ]:
interactions_df.reason_category.isnull().value_counts()

In [ ]:
interactions_df.was_escalated.unique()

In [ ]:
interactions_df.was_escalated.value_counts()

In [ ]:
interactions_df.channel.unique()

### Call transcripts table

In [ ]:
# Inspect call_transcripts schema
con.sql(f"DESCRIBE SELECT * FROM '{DATA_DIR}/call_transcripts/*.csv' LIMIT 1").show()

In [ ]:
transcripts_df = con.sql(f"SELECT * FROM '{DATA_DIR}/call_transcripts/*.csv' ").to_df()
transcripts_df.info()

In [ ]:
transcripts_df.customer_text.isnull().value_counts()

In [ ]:
transcripts_df.full_text.isnull().value_counts()

### Complaints table

In [ ]:
# Inspect complaints schema
con.sql(f"DESCRIBE SELECT * FROM '{DATA_DIR}/complaints/*.csv' LIMIT 1").show()

In [ ]:
complaints_df = con.sql(f"SELECT * FROM '{DATA_DIR}/complaints/*.csv' ").to_df()

In [ ]:
complaints_df.info()

In [ ]:
complaints_df.case_type.unique()

In [ ]:
complaints_df.case_type.isnull().value_counts()

In [ ]:
complaints_df.category.unique()

In [ ]:
complaints_df.category.isnull().value_counts()

## Column Value Distribution Analysis

In [ ]:
# Check contact reasons and escalation flags
con.sql(f"""
    SELECT reason_category, was_escalated, COUNT(*) as count
    FROM '{DATA_DIR}/call_center_interactions/*.csv'
    GROUP BY 1, 2
    ORDER BY count DESC
    LIMIT 20
""").show()

In [ ]:
# Check complaint categories and case types
con.sql(f"""
    SELECT case_type, category, COUNT(*) as count
    FROM '{DATA_DIR}/complaints/*.csv'
    GROUP BY 1, 2
    ORDER BY count DESC
    LIMIT 20
""").show()

## Join Verification & Class Balance Check

### Analising the relevant columns from tables call_center_interactions, call_transcripts and complaints to our ML model

In [ ]:
# From complaints table
print(f"complaints_df - #rows: {complaints_df.shape[0]}")
print("complaints_df.casetype: {complaints_df.case_type.unique().tolist()}")
print("complaints_df.category: {complaints_df.category.unique().tolist()}")
print(
    "complaints_df.complaint_id: "
    "{complaints_df.complaint_id.isnull().value_counts().to_dict()}"
)  # checking non null values

# From call_center_interactions table
print(f"call_center_interactions_df - #rows: {interactions_df.shape[0]}")
print(
    "call_center_interactions_df.contact_reason:"
    "{interactions_df.contact_reason.unique().tolist()}"
)
print(
    "call_center_interactions_df.was_escalated:"
    "{interactions_df.was_escalated.value_counts().to_dict()}"
)
print(
    "call_center_interactions_df.duration_seconds: "
    "{interactions_df.duration_seconds.isnull().value_counts().to_dict()}"
)  # checking non null values
print(
    "call_center_interactions_df.channel: {interactions_df.channel.unique().tolist()}"
)

# From call_center_transcripts
print(f"call_transcripts_df - #rows: {transcripts_df.shape[0]}")
print(
    "call_transcripts_df.customer_text: "
    "{transcripts_df.customer_text.isnull().value_counts().to_dict()}"
)  # checking non null values
print(
    "call_transcripts_df.detected_accent: "
    "{transcripts_df.detected_accent.unique().tolist()}"
)
print(
    "call_transcripts_df.detected_language: "
    "{transcripts_df.detected_language.unique().tolist()}"
)

### Merging the three datasets with Pandas

In [ ]:
print(interactions_df.interaction_id.dtype)
print(complaints_df.origin_interaction_id.dtype)
print(transcripts_df.interaction_id.dtype)

In [ ]:
# Strip whitespace and convert both key columns to string
interactions_df["interaction_id"] = (
    interactions_df["interaction_id"].astype(str).str.strip()
)
complaints_df["origin_interaction_id"] = (
    complaints_df["origin_interaction_id"].astype(str).str.strip()
)
transcripts_df["interaction_id"] = (
    transcripts_df["interaction_id"].astype(str).str.strip()
)

In [ ]:
import pandas as pd

# 1. First merge: transcripts + call center interactions (Inner Join)
# This keeps only transcripts that correspond to valid call center interactions
df_merged = pd.merge(
    transcripts_df,
    interactions_df,
    on="interaction_id",
    how="inner",
    suffixes=("_transcript", "_interaction"),
)

# Rename one to a clean 'customer_id' and drop the redundant column
df_merged["customer_id"] = df_merged["customer_id_interaction"]
df_merged = df_merged.drop(
    columns=["customer_id_interaction", "customer_id_transcript"]
)

In [ ]:
# Create a set of customer IDs who have filed at least one complaint
dispute_customer_ids = set(
    complaints_df[complaints_df.customer_id.notna()].customer_id.astype(str)
)

# Flag interactions belonging to historically high-dispute customers
df_merged["has_past_complaint"] = df_merged.customer_id.astype(str).isin(
    dispute_customer_ids
)

print(f"Total merged rows: {len(df_merged)}")
print(f"Total complaints linked: {df_merged.has_past_complaint.notna().sum()}")

In [ ]:
df_merged.columns.tolist()

In [ ]:
any(complaints_df.complaint_id.isna())

In [ ]:
all(complaints_df.origin_interaction_id.isna())

In [ ]:
complaints_df.columns

In [ ]:
interactions_df.columns

In [ ]:
transcripts_df.columns

In [ ]:
df_merged.columns

In [ ]:
def assign_target_label(row):
    contact_reason = str(row.get("contact_reason", "")).lower()
    was_escalated = row.get("was_escalated", False)

    # 1. Dispute Initiate: Transactions, complaints, or claim-related reasons
    if contact_reason in ["queja", "transaccional"]:
        return "dispute_initiate"

    # 2. Human Handoff: Calls flagged as escalated
    elif was_escalated:
        return "human_handoff"

    # 3. Account Inquiry: General product/technical inquiries (default)
    else:
        return "account_inquiry"


# Apply the label logic to your merged transcripts + interactions DataFrame
df_merged["label"] = df_merged.apply(assign_target_label, axis=1)

# Inspect target class distribution
print("Clean Target Class Distribution:")
print(df_merged["label"].value_counts())

In [ ]:
df = con.sql(f"""
    SELECT 
        t.customer_text AS text,
        i.reason_category,
        i.was_escalated,
        c.category AS complaint_category,
        c.case_type AS complaint_case_type
    FROM '{DATA_DIR}/call_transcripts/*.csv' t
    JOIN '{DATA_DIR}/call_center_interactions/*.csv' i
        ON t.interaction_id = i.interaction_id
    LEFT JOIN '{DATA_DIR}/complaints/*.csv' c
        ON i.interaction_id = c.origin_interaction_id
    WHERE t.customer_text IS NOT NULL AND LENGTH(TRIM(t.customer_text)) > 10
""").df()

# Display summary statistics
print(f"Total merged rows: {len(df)}")
df.head()

In [ ]:
df["complaint_category"].isnull().value_counts()

In [ ]:
df["complaint_case_type"].isnull().value_counts()

## Training the model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Features and target
X = df_merged["customer_text"]
y = df_merged["label"]

# Stratified split to keep class ratios balanced
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline definition
model_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(ngram_range=(1, 2), max_features=3000, sublinear_tf=True),
        ),
        (
            "clf",
            LogisticRegression(
                class_weight="balanced",  # Critical for the 31 human_handoff samples
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

# Train and evaluate
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

In [ ]:
df_merged.info()

## Multi-feature Baseline Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Pre-fill string text columns to avoid serialization issues
text_cols = [
    "full_text",
    "detected_keywords",
    "detected_intents",
    "main_topics",
]
for col in text_cols:
    df_merged[col] = df_merged[col].fillna("")

# Combine text fields into a single rich text feature
df_merged["rich_text"] = (
    df_merged["full_text"]
    + " "
    + df_merged["detected_keywords"]
    + " "
    + df_merged["detected_intents"]
    + " "
    + df_merged["main_topics"]
)

# Feature subsets
X_cols = [
    "rich_text",
    "channel",
    "detected_sentiment",
    "duration_seconds_interaction",
    "wait_time_seconds",
    "sentiment_score",
    "has_past_complaint",
]
X = df_merged[X_cols]
y = df_merged["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature processing components
text_transformer = TfidfVectorizer(
    ngram_range=(1, 2), max_features=3000, sublinear_tf=True
)

categorical_cols = ["channel", "detected_sentiment", "has_past_complaint"]
categorical_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

numeric_cols = [
    "duration_seconds_interaction",
    "wait_time_seconds",
    "sentiment_score",
]
numeric_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_transformer, "rich_text"),
        ("cat", categorical_transformer, categorical_cols),
        ("num", numeric_transformer, numeric_cols),
    ]
)

# Full pipeline
multi_feature_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "clf",
            LogisticRegression(
                class_weight="balanced", C=1.0, max_iter=1000, random_state=42
            ),
        ),
    ]
)

multi_feature_pipeline.fit(X_train, y_train)
y_pred = multi_feature_pipeline.predict(X_test)

print("=== MULTI-FEATURE CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=multi_feature_pipeline.classes_)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {c}" for c in multi_feature_pipeline.classes_],
    columns=[f"Pred {c}" for c in multi_feature_pipeline.classes_],
)

print("=== CONFUSION MATRIX ===")
print(cm_df)

In [ ]:
import sys

notebook_dir = Path.cwd()
project_root = notebook_dir.parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
from app.ml.data import assign_target_label, load_and_prepare_data

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
data_dir = (
    project_root / "app" / "data" / "sample"
)  # Update path to your data directory
df = load_and_prepare_data(data_dir)

print(f"Successfully loaded {len(df)} rows.")
print("Class Distribution:\n", df["label"].value_counts())